# Parte I: Análisis de datos (Pandas)

## 1. Ingesta de datos
Empezamos cargando los documentos CSV:

In [39]:
import pandas as pd
import re
import numpy as np

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
tags = pd.read_csv('tags.csv')
links = pd.read_csv('links.csv')

Comprobamos ahora para cada DataFrame el shape, columnas, dtype, head y conteo de nulos. Para ello creamos la función **comprobar_datos**.

In [17]:
def comprobar_datos(df, nombre):
    print(f'Información de: {nombre}')
    print(f'- Shape (filas, columnas):{df.shape}')
    print(f'- Columnas: {list(df.columns)}')
    print(f'- Tipos de datos (dtypes): {df.dtypes}')
    print(f'- Conteo de nulos: {dict(df.isnull().sum())}')
    print(f'- Primeras filas (head):\n{display(df.head())}')

comprobar_datos(movies, 'movies')
comprobar_datos(ratings, 'ratings')
comprobar_datos(tags, 'tags')
comprobar_datos(links, 'links')

Información de: movies
- Shape (filas, columnas):(9742, 3)
- Columnas: ['movieId', 'title', 'genres']
- Tipos de datos (dtypes): movieId     int64
title      object
genres     object
dtype: object
- Conteo de nulos: {'movieId': np.int64(0), 'title': np.int64(0), 'genres': np.int64(0)}


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


- Primeras filas (head):
None
Información de: ratings
- Shape (filas, columnas):(100836, 4)
- Columnas: ['userId', 'movieId', 'rating', 'timestamp']
- Tipos de datos (dtypes): userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object
- Conteo de nulos: {'userId': np.int64(0), 'movieId': np.int64(0), 'rating': np.int64(0), 'timestamp': np.int64(0)}


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


- Primeras filas (head):
None
Información de: tags
- Shape (filas, columnas):(3683, 4)
- Columnas: ['userId', 'movieId', 'tag', 'timestamp']
- Tipos de datos (dtypes): userId        int64
movieId       int64
tag          object
timestamp     int64
dtype: object
- Conteo de nulos: {'userId': np.int64(0), 'movieId': np.int64(0), 'tag': np.int64(0), 'timestamp': np.int64(0)}


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


- Primeras filas (head):
None
Información de: links
- Shape (filas, columnas):(9742, 3)
- Columnas: ['movieId', 'imdbId', 'tmdbId']
- Tipos de datos (dtypes): movieId      int64
imdbId       int64
tmdbId     float64
dtype: object
- Conteo de nulos: {'movieId': np.int64(0), 'imdbId': np.int64(0), 'tmdbId': np.int64(8)}


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


- Primeras filas (head):
None


## 2. Columna **year** desde el título

Implementamos la función **year_from_title** que devuelva un entero de cuatro cifras o valores ausentes si el título no sigue el patrón:

In [40]:
def year_from_title(title):
    match = re.search(r'\((\d{4})\)\s*$', str(title))
    if match:
        return match.group(1)
    else:
        return np.nan
    
movies['year'] = movies['title'].apply(year_from_title)
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')
movies['title'] = movies['title'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True).str.strip()

sin_ano = movies['year'].isna()
cantidad_sin_ano = sin_ano.sum()

print(f"Películas que NO tienen un año reconocible: {cantidad_sin_ano}")

print("\nCasos límite (muestra de títulos sin año):")
display(movies[sin_ano].head(10))

Películas que NO tienen un año reconocible: 13

Casos límite (muestra de títulos sin año):


,movieId,title,genres,year
6059,40697,Babylon 5,Sci-Fi,NaN
9031,140956,Ready Player One,Action|Sci-Fi|Thriller,NaN
9091,143410,Hyena Road,(no genres listed),NaN
9138,147250,The Adventures of Sherlock Holmes and Doctor W...,(no genres listed),NaN
9179,149334,Nocturnal Animals,Drama|Thriller,NaN
9259,156605,Paterson,(no genres listed),NaN
9367,162414,Moonlight,Drama,NaN
9448,167570,The OA,(no genres listed),NaN
9514,171495,Cosmos,(no genres listed),NaN
9515,171631,Maria Bamford: Old Baby,(no genres listed),NaN


## 3. Unificación de datos (merge/join)

Creamos la tabla película-usuario-rating. La clave que las va a unir es **movieId**

In [41]:
df_ratings_movies = pd.merge(ratings, movies, on='movieId', how='left')

print("Tabla película-usuario-rating resultante:")
display(df_ratings_movies.head(10))

Tabla película-usuario-rating resultante:


,userId,movieId,rating,timestamp,title,genres,year
0,1,1,4.0,964982703,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,1,3,4.0,964981247,Grumpier Old Men,Comedy|Romance,1995.0
2,1,6,4.0,964982224,Heat,Action|Crime|Thriller,1995.0
3,1,47,5.0,964983815,Seven (a.k.a. Se7en),Mystery|Thriller,1995.0
4,1,50,5.0,964982931,"Usual Suspects, The",Crime|Mystery|Thriller,1995.0
5,1,70,3.0,964982400,From Dusk Till Dawn,Action|Comedy|Horror|Thriller,1996.0
6,1,101,5.0,964980868,Bottle Rocket,Adventure|Comedy|Crime|Romance,1996.0
7,1,110,4.0,964982176,Braveheart,Action|Drama|War,1995.0
8,1,151,5.0,964984041,Rob Roy,Action|Drama|Romance|War,1995.0
9,1,157,5.0,964984100,Canadian Bacon,Comedy|War,1995.0


Se ha utilizado un left join (how='left') tomando como tabla principal (izquierda) el registro de eventos de los usuarios (ratings y tags respectivamente) y cruzándolas con el catálogo de películas (movies a la derecha) utilizando la clave movieId.

## 4. Agregaciones y segmentación

Para esta parte, usaremos la tabla unificada que creamos en el paso anterior (df_ratings_movies). Vamos a agrupar los datos por película y luego haremos una segmentación por décadas.

In [34]:
# 1. Agregación multi-columna (agg) agrupando por película
print("Agregación por Película (rating promedio y total de valoraciones):")
agg_peliculas = df_ratings_movies.groupby('title').agg(
    rating_promedio=('rating', 'mean'),
    total_valoraciones=('rating', 'count')
).sort_values('total_valoraciones', ascending=False) # Ordenamos para ver las más populares primero

display(agg_peliculas.head())

Agregación por Película (rating promedio y total de valoraciones):


,rating_promedio,total_valoraciones
title,,
Forrest Gump,4.164134,329
"Shawshank Redemption, The",4.429022,317
Pulp Fiction,4.197068,307
"Silence of the Lambs, The",4.161290,279
"Matrix, The",4.192446,278


In [42]:
# Primero creamos una nueva columna 'decada' usando división entera (ej: 1995 // 10 * 10 = 1990)
df_ratings_movies['decada'] = (df_ratings_movies['year'] // 10) * 10

print("Segmentación: Resumen de valoraciones por Década:")
agg_decadas = df_ratings_movies.groupby('decada').agg(
    rating_promedio=('rating', 'mean'),
    total_peliculas_distintas=('movieId', 'nunique'), # Cuántas películas únicas se valoraron
    total_valoraciones=('rating', 'count')
)

display(agg_decadas)

Segmentación: Resumen de valoraciones por Década:


,rating_promedio,total_peliculas_distintas,total_valoraciones
decada,,,
1900.0,3.312500,3,8
1910.0,3.312500,7,8
1920.0,3.740000,37,125
1930.0,3.733624,134,687
1940.0,3.870572,194,1101
1950.0,3.845011,277,1784
1960.0,3.808083,399,2858
1970.0,3.775676,498,4995
1980.0,3.518355,1175,12912


## 5. Preguntas sobre los datos

In [50]:
print("1. ¿Cuántas películas están listadas en movies?")
print(len(movies))

print("\n2. ¿Cuáles son las más antiguas?")
display(movies[movies['year'] == movies['year'].min()])

print("\n3. ¿Cuántas tienen 'Dracula' en el título (coincidencia parcial)?")
# Usamos str.contains. case=False hace que no distinga mayúsculas/minúsculas
print(movies['title'].str.contains('dracula', case=False, na=False).sum())

print("\n4. Títulos más comunes:")
display(movies['title'].value_counts().head(5))

print("\n5. Películas con 'Exorcist' ordenadas de más antigua a más moderna:")
exorcist_movies = movies[movies['title'].str.contains('exorcist', case=False, na=False)]
display(exorcist_movies.sort_values('year'))

print("\n6. ¿Cuántas con año 1950?")
print((movies['year'] == 1950).sum())

print("\n7. ¿Cuántas entre 1950 y 1959 inclusive?")
# between(1950, 1959) incluye ambos extremos automáticamente
print(movies['year'].between(1950, 1959).sum())

print("\n8. Año de la película con título exacto 'Batman':")
display(movies[movies['title'] == 'Batman'])
print("Contraste (otras películas que CONTIENEN 'Batman'):")
display(movies[movies['title'].str.contains('batman', case=False, na=False)].head())

print("\n9. Listado de películas que tienen como tag 'sci-fi' y 'adventure':")
# Sacamos los IDs de películas con sci-fi y los de adventure
id_scifi = set(tags[tags['tag'] == 'sci-fi']['movieId'])
id_adv = set(tags[tags['tag'] == 'adventure']['movieId'])

# Mostramos las películas que coinciden con esos IDs
display(movies[movies['movieId'].isin(id_scifi and id_adv)])

print("\n10. ¿Cuál es la tag más repetida?")
# value_counts() cuenta frecuencias, y head(1) saca la primera (la mayor)
display(tags['tag'].value_counts().head(1))

1. ¿Cuántas películas están listadas en movies?
9742

2. ¿Cuáles son las más antiguas?


,movieId,title,genres,year
5868,32898,"Trip to the Moon, A (Voyage dans la lune, Le)",Action|Adventure|Fantasy|Sci-Fi,1902.0



3. ¿Cuántas tienen 'Dracula' en el título (coincidencia parcial)?
9

4. Títulos más comunes:


title
Hamlet                   5
Three Musketeers, The    4
Jane Eyre                4
Misérables, Les          4
Christmas Carol, A       4
Name: count, dtype: int64


5. Películas con 'Exorcist' ordenadas de más antigua a más moderna:


...

,movieId,title,genres,year
126,153,Batman Forever,Action|Adventure|Comedy|Crime,1995.0
509,592,Batman,Action|Crime|Thriller,1989.0
1060,1377,Batman Returns,Action|Crime,1992.0
1174,1562,Batman & Robin,Action|Adventure|Fantasy|Thriller,1997.0
2418,3213,Batman: Mask of the Phantasm,Animation|Children,1993.0



9. Listado de películas que tienen como tag 'sci-fi' y 'adventure':


,movieId,title,genres,year
900,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,1981.0
2260,3000,Princess Mononoke (Mononoke-hime),Action|Adventure|Animation|Drama|Fantasy,1997.0
4445,6564,Lara Croft Tomb Raider: The Cradle of Life,Action|Adventure|Comedy|Romance|Thriller,2003.0
6208,45447,"Da Vinci Code, The",Drama|Mystery|Thriller,2006.0
7039,68954,Up,Adventure|Animation|Children|Drama,2009.0
7428,80834,Sintel,Animation|Fantasy,2010.0
8296,106489,"Hobbit: The Desolation of Smaug, The",Adventure|Fantasy|IMAX,2013.0
9692,184471,Tomb Raider,Action|Adventure|Fantasy,2018.0



10. ¿Cuál es la tag más repetida?


tag
In Netflix queue    131
Name: count, dtype: int64